# 📗 부록 — 평가셋을 직접 만들기

> **이 노트북은 수업 시간에 다루지 않는 참고 자료입니다.** 완주 기준에 들어가지 않습니다. 관심이 있을 때 읽어 보세요.

앞 시간에 우리는 **이미 만들어져 있는 평가셋**(`guide_eval_chunk.csv`)으로 검색을 쟀습니다. 그 파일에는 질문 20개와 각 질문의 정답 조각 id 가 적혀 있었지요. 그런데 그 라벨은 **누가, 어떻게** 붙인 걸까요?

실무에서 평가셋은 대개 **직접 만들어야 합니다.** 우리 회사 문서에 맞는 평가셋을 파는 곳은 없기 때문입니다. 이 부록은 그 과정을 보여 줍니다 — 후보를 좁히고, 모델에게 판정을 맡기되 **원문 인용을 강제**하고, 그 인용이 정말 조각 안에 있는지 **기계로 검증**하고, 마지막에 **사람이 확정**합니다.

그리고 만든 평가셋을 쓰기 전에 **점검**합니다. 평가셋이 틀리면 그것으로 잰 점수가 전부 함께 틀립니다. 그런데 화면에는 멀쩡해 보이는 숫자가 찍히기 때문에 알아채기가 어렵습니다.

**이 부록에서 하는 것**

- [ ] **구조화된 출력**으로 라벨 판정을 받고, 그 판정에 **원문 인용을 강제**한다
- [ ] 인용한 문장이 정말 그 조각 안에 있는지 **부분문자열로 검증**한다
- [ ] 기계가 통과시킨 라벨을 **사람이 다시 읽고** 확정한다
- [ ] 평가셋을 쓰기 전에 **일곱 항목**으로 점검한다

아래 준비 셀을 먼저 실행하세요(지난 시간과 같은 `.env` 의 `OPENAI_API_KEY` 를 씁니다).

> 이 노트북은 모델을 모두 **21회** 부릅니다 — 시연 판정 1회(후보 전체를 한 번에)와 전체 평가셋 재라벨링 20회(문항당 1회)입니다. 따라하기까지 직접 풀면 1회가 늘어 **22회**입니다. 어느 절에서 몇 회 부르는지는 그 자리에 적어 두었습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이번 시간 공통 부품 — 앞 시간에 배운 모델

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

### 다루는 자료 — 개인정보보호위원회 안내서

실제로 배포되고 있는 **공공 안내서 두 건**을 쪽 단위로 정리한 `guide_docs.csv` 입니다. 한 행이 안내서의 **한 쪽**이고, 지어낸 문서가 아닙니다.

| 원본 | 발간 | 원본 쪽수 |
|---|---|---|
| 개인정보보호위원회 「생성형 인공지능(AI) 개발·활용을 위한 개인정보 처리 안내서」 | 2025. 8. | 52 |
| 개인정보보호위원회 「개인정보 처리방침 작성지침(표준안)」 | 2026. 2. | 43 |

원본 두 건을 합치면 95쪽이지만 CSV 는 **74행**입니다. 표지·목차처럼 본문이 없는 쪽은 빼고 담았기 때문입니다.

> **출처 표시**: 두 안내서 모두 본문에서 *"무단전재를 금하며, 가공·인용할 때는 출처를 밝혀 주시기 바랍니다"* 라고 이용 조건을 밝히고 있습니다. 그래서 이 실습은 원본을 그대로 옮기지 않고 **파싱해 가공한 뒤 출처를 밝힙니다** — CSV 의 모든 행에 문서명과 쪽 번호가 붙어 있습니다. 자세한 내용은 `data/출처_평가코퍼스.md` 에 있습니다. **수업 밖으로 이 자료를 옮길 때도 같은 조건을 지켜 주세요.**

이 자료를 고른 이유는 분명합니다. 회사의 개인정보 담당자가 "우리 챗봇에 고객 대화를 학습에 써도 되나" 를 확인할 때 실제로 여는 문서이고, 40~50쪽이라 **"답이 어느 조각에 있나"** 가 지어낸 설정이 아니라 진짜 문제가 됩니다.

먼저 **검색 대상**인 문서 모음부터 훑어봅니다. 어떤 안내서가 몇 쪽씩 들어와 있는지 봅니다.

In [ ]:
# 검색 대상 문서를 먼저 훑어봅니다 -- 한 행이 안내서의 '한 쪽' 입니다.
#  검색에서 '문서 하나' 라고 부르는 것도 이 한 쪽을 가리킵니다.
import pandas as pd

docs = pd.read_csv('data/guide_docs.csv')
print('쪽 수:', len(docs))

# 본문은 한 쪽이 통째로 들어 있어 표에 넣으면 읽을 수 없습니다 -- 꼬리표 열만 먼저 봅니다.
display(docs[['id', '문서', '발간', '쪽', '소제목']].head(3))

# 본문은 따로 봅니다 -- 이 글자들이 조각으로 잘려 검색 대상이 됩니다.
print('[', docs.loc[0, 'id'], ']', docs.loc[0, '소제목'])
print(docs.loc[0, '본문'][:250])

In [ ]:
# 두 안내서에서 각각 몇 쪽씩 들어왔는지 셉니다.
#  한 안내서만 잔뜩 들어와 있으면 점수가 그 안내서의 성격만 따라갑니다.
print(docs['문서'].value_counts().to_string())
print('본문 평균 글자 수:', int(docs['본문'].str.len().mean()))

이번에는 점수를 매길 때 쓰는 **평가셋** 파일을 열어 봅니다. 한 문항은 세 가지로 이루어집니다 — `query`(질문) · `gold_chunks`(정답 라벨) · `근거문장`.

**근거 문장은 조각을 통째로 복사한 것이 아닙니다.** 그 조각 안에서 **답이 되는 대목만 원문 그대로 떼어 온 발췌**이고, **정답 조각 하나에 하나씩·같은 순서로** 짝지어 둡니다(조각은 `|`, 근거는 ` || ` 로 이어 붙입니다).

| | 무엇을 담나 | 무엇에 쓰나 |
|---|---|---|
| `gold_chunks` | 정답 **조각 id** | **채점** — 검색 결과 id 와 대조한다 |
| `근거문장` | 그 조각 안의 **원문 발췌** | **검수** — 라벨이 옳은지 사람이 확인하고, 색인을 다시 만들어도 그 문장을 찾아 라벨을 다시 붙인다 |

> 조각(400자 기준) 전체를 그대로 넣으면 "이 조각 어딘가에 답이 있다" 는 말을 되풀이하는 것이라 **검수 기능이 사라집니다.** 실제 값은 45~160자짜리 발췌입니다.

In [ ]:
# 평가셋 20문항을 불러옵니다 — 이 파일이 오늘 쓰는 '문제집' 입니다.
evalset = pd.read_csv('data/guide_eval_chunk.csv')
print(f'{len(evalset)}문항')

# 한 문항의 정답이 여럿일 수 있어 id 는 '|', 근거 문장은 ' || ' 로 이어 붙여 두었습니다.
display(evalset[['query_id', 'query', 'gold_chunks', '유형']].head(3))

In [ ]:
# 한 문항을 통째로 펼쳐 세 요소를 눈으로 확인합니다.
first_row = evalset.iloc[0]
print('질문      :', first_row['query'])
print('정답 라벨 :', first_row['gold_chunks'].split('|'))
print('근거 문장 :', first_row['근거문장'].split(' || ')[0][:60], '...')

---
# 1. 라벨은 왜 조각에 붙이나 — 색인부터 세운다

## 왜 중요할까요?
정답 라벨을 붙일 곳이 두 군데 있습니다. **문서 단위**("답은 안내서 33쪽에 있다")와 **조각 단위**("33쪽을 자른 조각 중 첫 번째에 있다")입니다. 문서 단위가 편해 보이지만, **검색기가 실제로 돌려주는 것은 조각**입니다. 검색 결과를 **문서 단위로 집계하면**(같은 문서에서 온 조각 여러 개를 문서 하나로 합쳐 세면) 어떤 실패가 가려지는지 우리 색인에서 직접 확인합니다.

<img src="images/문서vs청크_라벨.png" width="820">

*문서 단위로 집계하면, 정답이 없는 조각을 올려도 같은 문서라는 이유로 적중으로 계산됩니다.*

먼저 지난 시간의 파이프라인을 그대로 복원합니다. **자르기 → `Document` 로 감싸기 → 색인** 순서입니다.

> ⚠️ 자르는 규칙을 바꾸면 조각 id 가 전부 달라지고, 평가셋의 **정답 라벨이 어긋납니다.** 이 평가셋은 아래 규칙으로 `size=400` 으로 자른 조각에 라벨이 붙어 있습니다 — 그래서 여기서는 규칙을 그대로 씁니다.

In [ ]:
# 청킹 — 19일차에서 배운 그 스플리터입니다(문단 -> 줄 -> 문장 -> 낱말 순으로 큰 경계부터 존중합니다).
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size=400 : 안내서 한 쪽이 길어 반드시 잘립니다. chunk_overlap=80 은 경계에서 문장이 반 토막 나는 것을 막아 줍니다.
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
print('스플리터 준비 완료')

**자를 때 꼬리표(metadata)를 함께 답니다.** `부록_벡터DB_구축` 에서 쓴 규약 그대로입니다.

조각 id 는 꼬리표에 넣지 않습니다 — 아래에서 `ids=` 로 넘겨 **벡터DB 의 열쇠**로 삼고, 검색 결과에는 **`Document.id`** 로 실려 돌아오기 때문입니다(같은 값을 두 군데 적어 두면 한쪽만 고쳐질 뿐입니다).

> 여기서 만드는 색인은 **파일로 저장**합니다(`output/chroma_eval/`). 교안 02 가 여는 배포본 `data/chroma_day20` 의 **`guide_400`** 컬렉션이 **같은 자료·같은 규칙으로 만든 것**입니다 — 그래서 평가는 색인을 매번 다시 만들지 않고 **파일 하나를 공유해** 씁니다. 색인을 파일로 고정해야 조각 id 가 흔들리지 않고, 그래야 평가셋의 정답 라벨이 계속 유효합니다.

| 꼬리표 | 무엇 | 없으면 |
|---|---|---|
| `doc_id` | 어느 쪽에서 나왔나 | 문서 단위로 집계해 볼 수 없다 |
| `chunk_no` | 그 쪽의 몇 번째 조각인가 | 앞뒤 조각을 지목할 수 없다 |
| `chunk_total` | 그 쪽이 모두 몇 조각인가 | 문서 끝을 넘어가는 번호를 요구하게 된다 |
| `쪽` · `소제목` | 출처 | **라벨이 옳은지 사람이 확인할 수 없다** — 원문 어디를 펴야 할지 모른다 |

In [ ]:
# 안내서 74쪽을 모두 조각으로 자르면서 꼬리표를 함께 답니다.
from langchain_core.documents import Document

# 네 목록을 같은 순서로 채워 나갑니다 -- 뒤에서 zip 으로 묶어 쓰기 위해서입니다.
documents, chunk_ids, chunk_texts, chunk_docs = [], [], [], []
for row in docs.itertuples():
    parts = splitter.split_text(row.본문)     # 한 쪽이 여러 조각으로 잘립니다
    for i, piece in enumerate(parts):
        # i 는 그 쪽 안에서의 순번입니다(쪽이 바뀌면 다시 0).
        chunk_id = f'{row.id}-{i}'
        documents.append(Document(page_content=piece, metadata={
            'doc_id': row.id,            # 어느 쪽에서 나왔나 -- 문서 단위 집계에 쓴다
            'chunk_no': i,               # 그 쪽의 몇 번째 조각인가
            'chunk_total': len(parts),   # 그 쪽이 모두 몇 조각인가
            '쪽': int(row.쪽),           # 출처 -- 꼬리표에는 파이썬 기본 자료형만 담는다
            '소제목': row.소제목,
        }))
        chunk_ids.append(chunk_id)
        chunk_texts.append(piece)
        chunk_docs.append(row.id)

# 조각 본문을 id 로 바로 꺼내 쓸 수 있게 사전으로 만들어 둡니다(라벨을 눈으로 검수할 때 씁니다).
chunk_text = dict(zip(chunk_ids, chunk_texts))

print(f'문서 {len(docs)}개 -> 조각 {len(chunk_ids)}개')
print('앞 다섯 개 id:', chunk_ids[:5])
print('첫 조각의 꼬리표:', documents[0].metadata)

In [ ]:
# 조각을 임베딩해 색인합니다 -- 지난 시간에 쓴 그 부품 그대로입니다.
#  모델을 내려받고 조각 수백 개를 임베딩하느라 처음 한 번은 잠시 걸립니다.
#  색인은 이 노트북에서 가장 오래 걸리는 일이라 '한 번만' 만들고 끝까지 재사용합니다.
import shutil
from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 모델

# ids= 를 주면 벡터DB 안에서도 우리가 정한 조각 id 가 그대로 열쇠가 되고, 검색 결과의 d.id 로 돌아옵니다.
#  ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 문서가 중복되지 않습니다.
#  persist_directory 를 주면 폴더에 저장돼 커널을 꺼도 남습니다 -- 이것이 교안 02 가 여는 그 파일입니다.
#  (배포본을 덮어쓰지 않도록 여기서는 output/ 아래에 새로 만듭니다.)
TARGET_DIR = 'output/chroma_eval'
Path(TARGET_DIR).parent.mkdir(exist_ok=True)
if Path(TARGET_DIR).exists():
    shutil.rmtree(TARGET_DIR)      # 조각이 겹쳐 쌓이지 않게 통째로 다시 만듭니다

store_400 = Chroma.from_documents(documents, embeddings,
                                  collection_name='guide_400',
                                  persist_directory=TARGET_DIR, ids=chunk_ids)
print('색인 완료 — 조각', len(chunk_ids), '개 ->', TARGET_DIR)

이제 검색기를 만듭니다. 지난 시간에는 검색 결과의 **본문**을 프롬프트에 넣었지만, 평가에서는 **id** 가 필요합니다. 정답 라벨이 조각 id 이기 때문입니다. 그래서 검색 결과에서 **`d.id`**(색인할 때 `ids=` 로 넘긴 그 열쇠)만 뽑는 함수를 하나 만들어 둡니다.

함수가 **색인을 인자로 받는다**는 점을 눈여겨보세요. 색인을 바꿔 가며 재야 할 때 **재는 방법은 한 글자도 바꾸지 않고 색인만 갈아 끼우기** 위해서입니다.

In [ ]:
def search_ids(store, query, k):
    """질문과 가장 가까운 조각 k개의 id 를 순위 순서로 돌려준다."""
    # k 를 그때그때 바꿔 가며 재야 하므로 검색기는 부를 때마다 여기서 새로 만든다.
    #  검색기는 색인을 감싸기만 하는 얇은 껍데기라 새로 만들어도 느려지지 않는다.
    retriever = store.as_retriever(search_kwargs={'k': k})
    return [d.id for d in retriever.invoke(query)]


# 평가셋 첫 문항으로 시험해 봅니다 -- 정답과 나란히 놓고 보면 무엇을 재려는지가 분명해집니다.
print('질문   :', first_row['query'])
print('검색 5 :', search_ids(store_400, first_row['query'], 5))
print('정답   :', first_row['gold_chunks'].split('|'))

## 문서 단위로 집계하면 어떤 실패가 가려지나

다른 질문 하나로 **조각 단위**와 **문서 단위**를 나란히 봅니다.

In [ ]:
# 조각 단위 -- 검색이 올린 상위 3개 조각입니다.
credit_query = '사진이나 음성 파일에서 개인정보를 지울 때 규칙과 정규표현식만으로 충분한가요?'
top3 = search_ids(store_400, credit_query, 3)
for rank, chunk_id in enumerate(top3, 1):   # 두 번째 인자 1 -> 순위를 0 이 아니라 1 부터
    print(f'{rank}위 {chunk_id}')

# 같은 결과를 문서 단위로 집계합니다 -- 조각 id 의 앞부분이 문서 id 입니다(metadata['doc_id'] 와 같은 값).
doc_rank = []
for chunk_id in top3:
    doc_id = chunk_id.split('-')[0]
    # 같은 문서에서 온 조각이 여러 개여도 문서 순위에는 한 번만 넣습니다.
    if doc_id not in doc_rank:
        doc_rank.append(doc_id)

print('문서 순위:', doc_rank)

# 정답은 눈으로 적지 않고 **평가셋에서 꺼내 옵니다** -- 이 질문도 평가셋에 든 문항이기 때문입니다.
credit_row = evalset[evalset['query'] == credit_query].iloc[0]
credit_gold = credit_row['gold_chunks'].split('|')          # 정답 조각 id 목록
credit_gold_docs = {c.split('-')[0] for c in credit_gold}   # 그 조각들이 속한 문서
print(f"평가셋 문항 {credit_row['query_id']} 의 정답 조각: {credit_gold} (문서: {sorted(credit_gold_docs)})")

# 두 눈금을 같은 검색 결과에 대고 잽니다 -- 교집합이 비어 있지 않으면 맞힌 것입니다.
print('조각 단위로 맞혔나?', bool(set(top3) & set(credit_gold)))
print('문서 단위로 맞혔나?', bool(set(doc_rank) & credit_gold_docs))

In [ ]:
# 그런데 상위에 올라온 그 문서의 조각 본문을 실제로 읽어 봅니다(조각 하나를 통째로).
print('[검색이 올린 조각]')
print(chunk_text['ai33-2'])

In [ ]:
# 답이 실제로 들어 있는 조각은 따로 있습니다 -- 답이 되는 문장은 이 조각의 끝부분에 있습니다.
print('[답이 있는 조각]')
print(chunk_text['ai33-0'])

**문서로는 적중, 조각으로는 빗나감.**

`ai33-2` 는 노출된 개인정보의 삭제·차단 같은 다른 이야기를 하고 있어서 "규칙·정규표현식만으로 충분한가" 에 답하지 않습니다. 답은 `ai33-0` 의 마지막 문장 — *"규칙, 정규표현식 등을 통한 개인정보 검출 및 마스킹은 정확도 측면에서 한계가 있을 수 있으며, 이를 보완하기 위해 LLM 모델을 통해 …"* — 에 있고, 그 조각은 상위 3개에 올라오지 않았습니다.

문서 단위로 재면 이 문항은 **맞힌 것으로 셉니다.** 답이 없는 조각을 보여 주고도 점수를 받습니다. 그리고 답변을 만들 때 모델에게 전달되는 것은 **조각**입니다. 모델은 답할 근거가 없는 글을 받고도 무언가를 써 내려갑니다. 즉 **점수는 올라가는데 서비스는 실패하는** 상태입니다.

> **그래서 이 단원의 라벨은 조각 단위입니다.** 재는 단위를 검색기가 돌려주는 단위와 맞춥니다.

조각 단위에도 약점이 있습니다. 자르는 방법을 바꾸면 조각 id 가 전부 달라진다는 것입니다. 이 약점은 **근거 문장**으로 막습니다. "답은 `ai33-0` 에 있다" 대신 "답은 *'규칙, 정규표현식 등을 통한 …'* 이라는 문장에 있다" 로 적어 두는 것입니다. 그러면 색인을 다시 만들었을 때 그 문장을 품은 조각을 찾아 라벨을 **다시 붙일 수 있습니다.** 근거 문장을 함께 적어 두는 가장 큰 이유입니다.

## 정답 라벨이 지금 색인에 실제로 있는가

라벨은 조각 id 로 되어 있습니다. 그러니 **그 id 가 지금 색인에 실제로 있어야** 채점이 성립합니다. 없는 id 를 정답이라고 적어 두면 그 문항은 영원히 0점이 되는데, **에러는 나지 않습니다.** 조용히 틀립니다. 그래서 재기 전에 한 번 확인합니다.

In [ ]:
# 평가셋의 정답 id 가 전부 색인에 있는지 확인합니다.
#  색인에 든 id 를 집합으로 한 번에 모아 둡니다(문항마다 조회하면 느립니다).
indexed_ids = set(chunk_ids)

gold_all = [c for row in evalset.itertuples() for c in row.gold_chunks.split('|')]
missing = sorted({c for c in gold_all if c not in indexed_ids})

print('정답 라벨 총 개수 :', len(gold_all))
print('색인에 없는 id    :', missing)   # 비어 있어야 정상 -- 하나라도 있으면 그 문항은 조용히 0점이 된다
print('정답 라벨이 모두 색인에 있나?', len(missing) == 0)

### 🖐️ 함께 따라하기 — 다른 안내서에서 문서 단위로 집계해 보기

이번에는 **다른 안내서**(처리방침 표준안, `pp` 로 시작하는 문서)에서 같은 확인을 해 봅니다.

질문: **"만 14세 미만 아동의 개인정보를 처리할 때 법정대리인 동의는 어떻게 확인하나요?"**

1. 이 질문으로 상위 **5개** 조각의 id 를 찾으세요.
2. **1위 조각의 본문**을 출력해 답이 실제로 들어 있는지 눈으로 확인하세요.
3. 상위 5개의 조각 id 에서 **문서 id 만 떼어 내 중복을 없애고**(`ai33-2` → `ai33`), 서로 다른 문서가 **몇 개**인지 세어 출력하세요.

확인할 것: **조각 5개가 문서 5개가 아닙니다** — 한 문서에서 나온 조각 여러 개가 상위를 함께 차지하기 때문입니다. 채점을 이렇게 문서 단위로 하면 **조각 5개가 문서 3개로 줄어들면서, 답이 없는 조각을 올린 것도 "그 문서는 맞혔다" 로 계산됩니다.** 조각 단위로 재야 검색기가 실제로 무엇을 올렸는지가 그대로 드러납니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
fold_query = '만 14세 미만 아동의 개인정보를 처리할 때 법정대리인 동의는 어떻게 확인하나요?'
# 1) search_ids 로 상위 5개 조각 id 찾기 (색인은 store_400)
# 2) chunk_text 에서 1위 조각 본문 출력
# 3) 조각 id 앞부분(문서 id)만 모아 중복 없이 세기

---
# 2. 평가 데이터셋 구축 실전

평가셋 한 문항은 **질문 → 정답 라벨 → 근거 문장** 순서로 만들어집니다. 질문이 없으면 라벨을 붙일 대상이 없으니, **질문 만들기**부터 봅니다.

## 2-1. 질문은 어디서 오나

실무에서 쓰는 길은 크게 둘이고, **둘을 섞는 것**이 정석입니다.

| | 실제로 들어온 질문 | 문서에서 만들어 낸 질문 |
|---|---|---|
| 어디서 | 상담·문의 로그, 사내 문의 채널, 기존 FAQ, 검색어 기록 | 문서 조각을 모델에게 주고 "이 대목으로 답할 수 있는 질문을 써라" 고 시킨다 |
| 장점 | **사용자가 실제로 쓰는 말투와 관심사**가 그대로 담긴다 | 빠르고 많이 만든다. 문서 구석까지 **골고루** 덮는다 |
| 약점 | 모으는 데 시간이 든다. 서비스 초기에는 아예 없다 | 문서 문장을 그대로 베끼기 쉽고, 말투가 사용자와 다르다 |
| 답이 있는지 | **모를 수도 있다**(그래서 답 없는 질문도 평가셋에 넣을 수 있다) | 만들 때부터 정답 조각이 정해져 있다 |

> **문서에서 만든 질문에는 함정이 하나 있습니다.** 조각의 문장을 거의 그대로 옮겨 질문을 만들면 검색이 글자만 보고도 맞히게 되어 **점수가 실제보다 높게 나옵니다.** 그래서 만든 질문은 사람이 **말을 바꿔 다시 쓰고**, 원문과 얼마나 겹치는지 재서 걸러 냅니다(3절 점검 항목).

그리고 **질문을 검색으로만 뽑아 내지 않습니다.** 지금 검색기가 못 찾는 대목은 후보에도 들어오지 못해, 평가셋이 **지금 검색기가 잘하는 쪽으로 기웁니다.** 문서를 직접 읽고 쓴 질문을 섞어야 그 치우침이 줄어듭니다.

**좋은 질문의 조건** — 넷 다 만족해야 그 문항이 무언가를 말해 줍니다.

1. **사용자의 말투로 쓴다** — 문서 제목이 아니라 "…하려면 무엇을 해야 하나요?" 처럼
2. **답이 자료 안에 있다** — 없는 질문만 모으면 검색기가 아니라 문서를 평가하는 셈이 된다
3. **난이도를 섞는다** — 한 조각으로 끝나는 질문과 **여러 조각에 흩어진 질문**을 함께 넣는다(전부 단일 정답이면 Recall 이 Hit 과 같은 값이 되어 지표 넷 중 둘만 보게 된다)
4. **같은 것을 두 번 묻지 않는다** — 표현만 다른 중복 질문은 그 주제에 가중치를 몰래 준다

몇 개나 필요할까요. **20문항이면 한 문항이 5%p** 를 흔듭니다. 수업·프로토타입은 20~30문항으로 시작하고, 판단의 근거로 쓸 평가셋은 보통 100문항 이상으로 키웁니다.

지금 쓰는 평가셋도 이 방식으로 만들어졌습니다 — **20문항 중 사람이 쓴 것 14개, 문서에서 만든 것 6개**이고, 정답이 여럿인 문항이 15개입니다(`질문출처`·`유형` 열에 적혀 있습니다).

## 2-2. 라벨을 붙인다 — 그리고 검증한다

질문이 모였으면 각 질문의 답이 **어느 조각에** 있는지 정해야 합니다. 조각이 300개 가까운데 질문마다 그것을 다 읽을 수는 없습니다. 실무에서는 이렇게 합니다.

| 단계 | 하는 일 | 왜 |
|---|---|---|
| 1단계 후보 좁히기 | 그 질문으로 검색해 상위 10~15개만 남긴다 | 사람이 읽을 수 있는 분량으로 줄인다 |
| 2단계 판정 | 후보마다 "이 조각만으로 답할 수 있나" 를 묻는다 | '정답' 의 기준을 한 문장으로 못박는다 |
| 3단계 근거 인용 | 답할 수 있다면 **그 문장을 원문 그대로** 옮겨 적게 한다 | 판정의 이유가 남는다 |
| 4단계 검증 | 인용한 문장이 조각 안에 실제로 있는지 기계로 확인 | 지어낸 근거를 걸러 낸다 |

<img src="images/라벨링_4단계.png" width="820">

*후보 좁히기 → 판정 → 근거 인용 → 검증. 검증에서 걸린 라벨은 삭제하지 않고 사람 확인 목록으로 보냅니다.*

**2·3단계는 사람이 해도 되고 모델에게 시켜도 됩니다.** 사람이 하면 정확하지만 느립니다 — 문항 20개에 후보 12개면 읽어야 할 조각이 240개입니다. 모델은 그것을 몇 분에 끝내는 대신 **틀린 판정을 섞습니다.**

그래서 실무의 표준은 **모델이 초안을 만들고 사람이 확정하는 것**입니다. 모델에게 판정과 근거 인용을 맡기고, 4단계에서 **인용한 문장이 그 조각 안에 실제로 있는지 글자로 대조**해 명백한 오류를 기계가 먼저 걷어 낸 뒤, 남은 것만 사람이 읽고 확정합니다. **사람이 읽어야 할 분량을 줄이는 것**이지 사람을 빼는 것이 아닙니다.

**4단계 검증은 두 가지를 걸러 내는데, 하나는 걸러야 할 것이고 하나는 아닙니다.**

| 검증에서 탈락 | 실제로는 | 어떻게 하나 |
|---|---|---|
| 모델이 **지어낸 근거** | 틀린 라벨 | 버린다 — 검증의 목적이 이것이다 |
| 맞는 근거인데 **글자가 다름**(표·`▲` 같은 기호를 빼고 옮겨 적은 경우) | 맞는 라벨 | 버리면 손해다 — **사람 확인 목록**(`review_needed`)으로 보낸다 |

> **후보에 없으면 라벨도 없습니다.** 검색이 상위로 올리지 못한 조각은 판정 대상에 아예 들어오지 못해, 정답인데도 라벨이 붙지 않습니다. 그래서 후보를 넉넉히(10~15개) 잡습니다.


In [ ]:
# 1단계 후보 좁히기 -- 질문 하나로 상위 조각만 남깁니다.
label_query = '개인정보 처리방침을 만들지도, 공개하지도 않으면 어떤 제재를 받나요?'
# 12개 -- 사람이 읽어 낼 만하면서 정답을 빠뜨리지 않을 만큼 넉넉한 수입니다.
candidates = search_ids(store_400, label_query, 12)
print(candidates)

## 2~3단계 — 판정을 **구조화된 출력**으로 받는다

후보마다 두 가지를 알아야 합니다. **이 조각으로 답할 수 있는가**와 **답이 되는 문장은 무엇인가**입니다. 이런 답을 줄글로 받으면 곤란합니다 — "네, 답할 수 있습니다. 근거는…" 에서 참/거짓을 뽑아내려면 글자를 뒤져야 하고, 모델이 말투를 조금만 바꿔도 깨집니다.

앞서 배운 **구조화된 출력**이 이 자리를 위한 도구입니다. 받을 모양을 **스키마**로 선언하고 `model.with_structured_output(스키마)` 로 씌우면 결과가 **파이썬 객체**로 돌아옵니다.

**스키마의 모양은 답의 모양을 따라갑니다.** 한 질문의 정답 조각은 **하나가 아니라 여럿**입니다(이 평가셋도 20문항 중 15문항이 복수 정답입니다). 그러니 후보를 하나씩 따로 묻는 대신, **후보 목록을 한 번에 주고 답이 되는 조각들을 목록으로 받습니다.**

| | 조각 하나씩 묻기 | 후보를 한 번에 주고 목록으로 받기 |
|---|---|---|
| 모델 호출 | 후보 수만큼(12회) | **1회** |
| 스키마 | 조각 하나의 판정 | **`(조각 id, 근거 문장)` 의 목록** |
| 정답이 여럿일 때 | 결과를 우리가 모아 붙여야 한다 | **답의 모양 그대로** 돌아온다 |
| 주의 | — | 모델이 **조각 여럿을 합쳐** 답이 된다고 볼 수 있다 → 프롬프트로 막는다 |

그래서 스키마를 **두 겹**으로 만듭니다. 안쪽은 조각 하나의 라벨(`ChunkLabel`), 바깥쪽은 그 목록(`Labels`)입니다. `Field(description=...)` 에 적는 설명은 **모델에게 가는 지시**입니다 — "조각 원문 그대로"·"후보 목록에 있는 id 만" 같은 요구를 여기에 적습니다.

In [ ]:
# 판정 결과로 받을 '모양' 을 스키마로 선언합니다 -- 답이 여럿이므로 목록입니다.
from pydantic import BaseModel, Field


class ChunkLabel(BaseModel):
    """정답 조각 하나와 그 근거."""

    chunk_id: str = Field(description='답이 되는 조각의 id. 후보 목록에 있는 id 만 쓴다')
    evidence: str = Field(description='그 조각에서 답이 되는 문장을 원문 그대로 옮긴 것')


class Labels(BaseModel):
    """한 질문의 라벨 전체."""

    labels: list[ChunkLabel] = Field(
        description='질문에 답할 수 있는 조각들. 하나도 없으면 빈 목록')


# 모델에 스키마를 씌워 둡니다 -- 이 판정기는 아래에서 계속 재사용합니다.
judge_model = model.with_structured_output(Labels)
print('판정 스키마 준비:', list(Labels.model_fields), '/', list(ChunkLabel.model_fields))

In [ ]:
# 판정 프롬프트를 템플릿으로 만듭니다 -- 질문과 후보 묶음만 갈아 끼우며 같은 지시로 돌립니다.
from langchain_core.prompts import ChatPromptTemplate

judge_prompt = ChatPromptTemplate.from_template('''[질문]
{question}

[후보 조각]
조각마다 맨 앞 줄에 id 가 있고, 조각과 조각은 구분선(-----)으로 나뉩니다.

{chunks}

[규칙]
1. 각 조각을 따로 보고, **그 조각 하나만으로** 위 질문에 답할 수 있는 조각만 고릅니다.
   조각 둘을 합쳐야 답이 되는 경우는 고르지 않습니다.
2. 고른 조각마다 chunk_id 에는 그 조각의 id 를 **그대로 옮겨 적습니다**(새로 만들지 않습니다).
3. evidence 에는 그 조각에서 답의 근거가 되는 문장을 **원문 그대로** 옮겨 적습니다.
4. 답할 수 있는 조각이 하나도 없으면 빈 목록으로 답합니다.''')

# 템플릿 | 스키마 씌운 모델 -- 자리를 채우면 라벨 목록이 나오는 하나의 부품이 됩니다.
judge_chain = judge_prompt | judge_model


def format_candidates(chunk_ids):
    """후보 조각들을 id 를 붙이고 구분선으로 나눈 한 덩어리 글로 만든다."""
    # 모델이 id 로 답하려면 본문과 id 가 눈에 띄게 짝지어 보여야 한다.
    blocks = [f'id: {cid}\n{chunk_text[cid]}' for cid in chunk_ids]
    return ('\n' + '-' * 60 + '\n').join(blocks)


def label_question(query, chunk_ids):
    """후보 조각들 중 질문에 답할 수 있는 것을 골라 (id, 근거 문장) 목록으로 받는다."""
    return judge_chain.invoke({'question': query, 'chunks': format_candidates(chunk_ids)})


# 모델에게 실제로 어떤 글이 가는지 앞부분만 확인해 봅니다.
print(format_candidates(candidates[:2])[:300])

In [ ]:
# 후보 12개를 한 번에 판정합니다(모델 호출 1회).
verdict = label_question(label_query, candidates)
print('돌아온 것의 종류:', type(verdict).__name__)   # 문자열이 아니라 Labels 객체
print('고른 조각 수    :', len(verdict.labels))
for item in verdict.labels:
    print(f'  {item.chunk_id} :: {item.evidence}')

## 4단계 — 인용한 문장이 조각 안에 실제로 있는가

모델이 "근거는 이 문장입니다" 라고 답했다고 그것이 조각에 있는 문장이라는 보장은 없습니다. 그래서 **기계로 확인**합니다. 줄바꿈·띄어쓰기만 다른 인용은 통과시켜야 하므로 양쪽에서 공백을 지우고 비교합니다.

In [ ]:
import re


def squeeze(text):
    """공백(띄어쓰기·줄바꿈)을 모두 지운다 -- 표기 차이 때문에 검증이 실패하지 않게."""
    return re.sub(r'\s+', '', text)


def quoted_in_chunk(sentence, chunk_id):
    """인용 문장이 그 조각 본문 안에 실제로 있는지 확인한다."""
    if not sentence.strip():          # 빈 문자열은 어떤 본문에도 '들어 있다' 가 되어 버린다
        return False
    return squeeze(sentence) in squeeze(chunk_text[chunk_id])



# 방금 받은 라벨 하나로 검증해 봅니다.
first = verdict.labels[0]
print('인용 검증          :', quoted_in_chunk(first.evidence, first.chunk_id))
print('지어낸 문장으로 검증:', quoted_in_chunk('이 조각은 과태료 5천만원을 규정한다.', first.chunk_id))

이제 **모델이 고른 라벨을 하나씩 검증**해 확정합니다. 모델 호출은 이미 끝났고(위에서 1회), 여기서는 돌아온 목록을 검사만 합니다. 판정과 검증을 **둘 다** 통과한 조각만 정답 라벨이 됩니다.

In [ ]:
# 모델이 고른 라벨이 두 갈래로 갈립니다 -- 아래 if 가 그 갈래입니다.
#  (1) 인용이 원문과 같다 -> 정답 라벨로 확정
#  (2) 인용이 원문과 다르다 -> 버리지 않고 사람이 볼 목록으로.
gold, evidence, review_needed = [], [], []
for item in verdict.labels:
    # 모델이 후보에 없는 id 를 지어낼 수도 있다 -- 먼저 걸러 낸다.
    if item.chunk_id not in candidates:
        review_needed.append(item.chunk_id)
    elif quoted_in_chunk(item.evidence, item.chunk_id):
        gold.append(item.chunk_id)
        evidence.append(item.evidence)
    else:
        review_needed.append(item.chunk_id)

print(f'정답 조각 {len(gold)}개 / 사람이 다시 볼 것 {len(review_needed)}개')
for chunk_id, sentence in zip(gold, evidence):
    print(f'  {chunk_id} :: {sentence}')

In [ ]:
# 파일에 들어 있는 같은 질문의 라벨과 견줍니다.
saved = evalset[evalset['query_id'] == 'guide09'].iloc[0]
print('파일의 라벨 :', saved['gold_chunks'].split('|'))
print('방금 만든 것:', gold)

방금 만든 라벨과 파일의 라벨이 다를 수 있습니다. 파일을 만들 때는 **다른 모델**로 판정했기 때문입니다. 같은 절차를 밟아도 **누가 판정하느냐에 따라 라벨이 달라집니다.** 그래서 최종 라벨은 **파일로 고정해 두고**(더 이상 바꾸지 않고) 그 파일로만 점수를 잽니다. 매번 다시 판정하면 지난번 점수와 이번 점수를 견줄 수 없습니다.

이 절차로 20문항 전체에 라벨을 붙여 둔 것이 `guide_eval_chunk.csv` 입니다. 만드는 동안 실제로 있었던 일 세 가지를 적어 둡니다.

- **모델이 답을 못 찾은 문항이 있었습니다.** 후보 어느 것도 "답할 수 있음" 이 아니어서 그 문항은 평가셋에서   빠졌습니다(그래서 문항 번호가 `guide02` 부터 시작합니다). 사람이 다시 보고 살릴지 버릴지 정해야 하는 자리입니다.
- **작은 모델은 눈에 보이는 답도 놓쳤습니다.** 1절에서 본 `ai33-0`(규칙·정규표현식의 한계를 적은 조각)을   더 작은 모델은 "답할 수 없음" 으로 판정했습니다. 사람이 읽으면 바로 보이는 문장입니다.
- **반대 방향의 실수도 있었습니다.** 어떤 문항에서는 모델이 정답이 아닌 조각까지 정답으로 넣었습니다.   질문과 관련은 있지만 질문에 답하지는 않는 조각입니다. 예를 들어 무엇을 적어야 하는지 나열하기만 한   **목차 표**를 "파기 방법을 구분해 적어야 하나" 의 근거로 삼았습니다. 사람이 읽고 라벨 **4건**을 뺐습니다.

> 자동 판정은 **후보를 줄여 주는 도구**입니다. 라벨을 확정하는 것은 사람입니다. 평가셋은 다른 모든 수치의 기준점이므로, 여기가 틀리면 그다음 모든 판단이 함께 틀립니다.

### 🖐️ 함께 따라하기 — 다른 질문에 라벨 붙여 보기

다른 질문으로 라벨을 만들어 봅니다. **모델 호출 1회**입니다(후보를 한 번에 넘깁니다).

질문: **"영상정보처리기기를 설치했을 때 처리방침에 적어야 하는 항목은 무엇인가요?"**

1. 상위 **5개** 조각을 후보로 좁히세요.
2. `label_question` 으로 후보를 한 번에 판정하고, 돌아온 라벨마다 `quoted_in_chunk` 로 인용을 검증해 통과한 것만 남기세요.
3. 남은 조각 id 와 근거 문장을 출력하세요.

확인할 것: 정답 조각이 몇 개 나왔는지. 하나가 아닐 수 있습니다 — **답이 여러 조각에 나뉘어 있는 질문**은 실제로 흔합니다. 그리고 후보를 5개로 줄였으니 **더 아래에 있던 정답은 후보에도 못 들어왔다**는 점을 기억하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
follow_label_q = '영상정보처리기기를 설치했을 때 처리방침에 적어야 하는 항목은 무엇인가요?'
# 1) search_ids 로 상위 5개 후보 좁히기 (색인은 store_400)
# 2) label_question 으로 한 번에 판정하고 quoted_in_chunk 로 검증한 것만 남기기
# 3) 남은 조각 id 와 근거 문장 출력

### 전체 평가셋에 한꺼번에 돌려 보기

문항 하나로 밟아 본 절차(**후보 좁히기 → 판정 → 인용 검증**)를 **평가셋 20문항 전체**에 그대로 돌립니다. 문항마다 모델 호출 1회이므로 **20회**입니다(1~2분).

돌린 뒤에 볼 것은 점수가 아니라 **파일의 라벨과 어디서 갈리는가** 입니다. 파일은 다른 모델로 판정해 만들었으므로 같은 절차를 밟아도 라벨이 똑같기는 어렵습니다. 겹치는 것 · 이번에만 나온 것 · 파일에만 있는 것을 나란히 놓고 **어느 쪽이 맞는지 사람이 읽어 정하는 것**이 라벨링의 마지막 단계입니다.

In [ ]:
# 전체 20문항을 같은 절차로 다시 라벨링합니다 (모델 호출 20회 -- 1~2분 걸립니다).
K_CAND = 12   # 문항마다 후보로 넘길 조각 수 -- 위 시연과 같은 값입니다.

# 1) 먼저 문항마다 후보를 좁혀 '판정 입력' 만 만들어 둡니다 (여기서는 모델을 부르지 않습니다).
#    검색은 값이 정해져 있어 빠르고 공짜입니다 -- 돈이 드는 판정은 아래에서 한꺼번에 합니다.
judge_inputs, cand_by_q = [], {}
for row in evalset.itertuples():
    cands = search_ids(store_400, row.query, K_CAND)
    cand_by_q[row.query_id] = cands          # 나중에 '후보에 없는 id' 를 가려내는 데 쓴다
    judge_inputs.append({'question': row.query, 'chunks': format_candidates(cands)})

# 2) 한꺼번에 판정합니다 -- batch 는 여러 호출을 동시에 보내 줍니다(하나씩 부르는 것보다 훨씬 빠릅니다).
#    max_concurrency 로 동시에 나가는 수를 묶어 둡니다(속도 제한에 걸리지 않게).
verdicts = judge_chain.batch(judge_inputs, config={'max_concurrency': 4})

# 3) 앞에서 만든 갈래 나누기를 문항마다 되풀이합니다 -- 인용 검증을 통과한 것만 라벨로 남깁니다.
compare_rows = []
for row, verdict in zip(evalset.itertuples(), verdicts):
    cands = cand_by_q[row.query_id]
    new_gold, needs_review = [], []
    for item in verdict.labels:
        if item.chunk_id not in cands:
            needs_review.append(item.chunk_id)          # 후보로 준 적 없는 id -- 조각이 없다
        elif quoted_in_chunk(item.evidence, item.chunk_id):
            new_gold.append(item.chunk_id)              # 근거가 원문에 실재한다 (라벨 초안)
        else:
            needs_review.append(item.chunk_id)          # 인용이 원문과 다르다 -- 사람이 볼 몫

    # 파일의 라벨과 견줍니다. 집합으로 바꿔 '겹침 / 이번에만 / 파일에만' 으로 가릅니다.
    saved_gold = row.gold_chunks.split('|')
    same = set(new_gold) & set(saved_gold)
    union = set(new_gold) | set(saved_gold)
    compare_rows.append({
        'query_id': row.query_id,
        '파일': len(saved_gold),
        '이번': len(new_gold),
        '겹침': len(same),
        '이번에만': '|'.join(sorted(set(new_gold) - set(saved_gold))) or '-',
        '파일에만': '|'.join(sorted(set(saved_gold) - set(new_gold))) or '-',
        # 겹침비율(자카드) -- 1.0 이면 완전히 같은 라벨, 0.0 이면 하나도 안 겹친다.
        '겹침비율': round(len(same) / len(union), 2) if union else 0.0,
        '사람확인': len(needs_review),
    })

compare = pd.DataFrame(compare_rows)
display(compare)

In [ ]:
# 전체를 한눈에 -- 판정자가 다르면 라벨이 얼마나 갈리는지 봅니다.
print(f'문항 수                 : {len(compare)}')
print(f'파일과 완전히 같은 문항 : {(compare["겹침비율"] == 1.0).sum()}개')
print(f'겹침비율 평균           : {compare["겹침비율"].mean():.2f}')
print(f'이번에만 나온 라벨      : {sum(r != "-" for r in compare["이번에만"])}문항')
print(f'파일에만 있던 라벨      : {sum(r != "-" for r in compare["파일에만"])}문항')
print(f'인용 검증에서 걸러진 것 : {compare["사람확인"].sum()}개 (사람이 조각을 열어 대조할 몫)')

# 가장 많이 갈린 문항을 열어 봅니다 -- 어느 쪽이 맞는지는 조각을 읽어야 정해집니다.
worst = compare.sort_values('겹침비율').iloc[0]
print(f'\n가장 많이 갈린 문항: {worst["query_id"]}')
print('질문      :', evalset[evalset['query_id'] == worst['query_id']].iloc[0]['query'])
print('이번에만  :', worst['이번에만'])
print('파일에만  :', worst['파일에만'])

돌려 보면 **파일과 완전히 같은 문항은 몇 개 되지 않고, 겹침비율 평균도 0.5 를 밑돕니다.** 판정 모델이 다르고(`gpt-4o-mini` 는 "이 조각 하나만으로 답이 되는가" 를 꽤 엄격하게 봅니다), 후보를 `K_CAND` 개로 좁혔기 때문입니다. `파일에만` 에 남은 조각 중에는 **후보에 못 들어와 판정 기회조차 없던 것**이 섞여 있습니다 — `K_CAND` 를 늘리면 그 몫이 줄지만 모델에 넣는 글이 길어져 비용이 올라갑니다. 남은 라벨이 아예 0개인 문항도 나올 수 있습니다.

여기서 볼 것은 "라벨이 갈렸다" 가 아니라 **갈린 문항이 곧 사람이 읽어야 할 목록**이라는 점입니다. 스무 문항을 전부 읽는 대신 갈린 곳만 열어 어느 쪽이 맞는지 정하면 됩니다. 자동 라벨링은 **사람의 일을 없애는 것이 아니라 줄이는 것**입니다.

---
# 3. 평가셋을 점검한다

## 왜 필요할까요?
라벨이 붙었다고 끝이 아닙니다. 평가셋에 결함이 있으면 **지표가 조용히 거짓말을 합니다** — 숫자는 멀쩡하게 나오는데 그 숫자가 아무것도 말해 주지 않습니다. 내보내기 전에 점검할 항목입니다.

| 점검 | 통과 못 하면 |
|---|---|
| 1. 정답 조각 id 가 색인에 실제로 있는가 | 채점이 조용히 0점 처리된다 |
| 2. 근거 문장이 그 조각 원문에 있는가 | 라벨의 근거가 지어낸 것이다 |
| 3. 질문이 문서를 베끼지 않았는가 | 점수가 실제보다 높게 나온다 |
| 4. 정답이 여럿인 문항이 섞여 있는가 | Recall 이 Hit 과 늘 똑같은 값이 된다 |
| 5. 지금 검색기가 못 찾는 문항이 있는가 | 전부 만점이라 좋아졌는지 나빠졌는지 알 수 없다 |
| 6. 한 질문의 정답 조각이 지나치게 많지 않은가 | K 를 뭘 줘도 Recall 이 오르지 않는다 |
| 7. 네 지표가 실제로 서로 다른 값을 내는가 | 지표를 넷 만들어 놓고 사실은 둘만 보고 있게 된다 |

> 표에 나오는 지표 이름(Hit@K·Precision@K·Recall@K·MRR)은 **앞 시간에 배운 그것들입니다.** 여기서는 "평가셋이 이런 모양이면 지표가 서로 겹친다" 는 것만 알아 두면 됩니다.

4번을 조금 더 설명합니다. 모든 문항의 정답이 하나뿐이면 Recall@K 는 Hit@K 와 **항상 같아집니다.** Precision@K 도 Hit@K 를 K 로 나눈 값이 됩니다. 지표를 넷 만들어 놓고 실제로는 두 가지만 보고 있는 셈입니다. 그래서 **정답이 둘 이상인 문항을 일부러 넣습니다.**

In [ ]:
# 1~2번 점검 -- 정답 조각 id 와 근거 문장이 실제로 있는지 확인합니다.
#  두 항목을 한 번에 도는 이유: 라벨 한 줄이 (조각 id, 근거 문장) 한 쌍이라
#  같은 자리에서 '조각이 있나 -> 그 조각에 그 문장이 있나' 순서로 이어 보는 것이 자연스럽습니다.
bad_id, bad_evidence = [], []          # 1번에 걸린 것 / 2번에 걸린 것을 따로 모읍니다
for row in evalset.itertuples():
    # 파일에는 한 칸에 여러 개가 이어 붙어 있습니다 -- id 는 '|', 근거 문장은 ' || ' 로 나눕니다.
    ids = row.gold_chunks.split('|')
    sentences = row.근거문장.split(' || ')
    if len(ids) != len(sentences):
        # 개수가 어긋나면 짝이 밀려 엉뚱한 조각에서 문장을 찾게 됩니다. 먼저 알려 줍니다.
        print(f'라벨과 근거 개수가 다릅니다: {row.query_id}')   # zip 은 짧은 쪽에서 조용히 끊는다
    for chunk_id, sentence in zip(ids, sentences):
        if chunk_id not in indexed_ids:
            # 1번 -- 색인에 없는 조각을 정답이라 적어 둔 것. 채점은 에러 없이 조용히 0점이 됩니다.
            bad_id.append(chunk_id)
        elif not quoted_in_chunk(sentence, chunk_id):
            # 2번 -- 조각은 있는데 근거 문장이 그 원문에 없다. 지어냈거나 옮겨 적다 달라진 것입니다.
            #  elif 인 이유: 조각이 없으면 그 안에서 문장을 찾는 일 자체가 성립하지 않습니다.
            bad_evidence.append(chunk_id)

# 둘 다 빈 목록이어야 정상입니다 -- 하나라도 있으면 그 문항의 점수를 믿을 수 없습니다.
print('없는 조각 id     :', bad_id)
print('원문에 없는 근거 :', bad_evidence)

In [ ]:
# 3번 점검 -- 질문이 문서를 베끼지 않았는가. 정답 조각들과의 겹침 평균을 문항마다 잽니다.
#  겹침은 '세 글자 덩어리' 기준입니다. 두 글자로 세면 한국어 조사·어미가 자주 겹쳐
#  베끼지 않은 질문까지 0.6 을 넘깁니다(실측) -- 멀쩡한 문항이 베낀 것으로 걸립니다.
def overlap_ratio(query, text):
    """질문의 세 글자 덩어리 중 몇 %가 문서 본문에 그대로 나오는가."""
    # 띄어쓰기만 다른 표현도 같은 것으로 세도록 공백을 모두 지웁니다.
    squeezed = re.sub(r'\s+', '', query)
    # 세 글자씩 한 칸 밀며 잘라, 중복 없이 모읍니다.
    grams = {squeezed[i:i + 3] for i in range(len(squeezed) - 2)}
    body = re.sub(r'\s+', '', text)
    return sum(1 for g in grams if g in body) / len(grams)


def query_overlap(row):
    """한 문항의 질문이 그 문항의 정답 조각들을 얼마나 베꼈는지의 평균."""
    ids = row.gold_chunks.split('|')
    return sum(overlap_ratio(row.query, chunk_text[c]) for c in ids) / len(ids)


evalset['겹침'] = [query_overlap(row) for row in evalset.itertuples()]
print(evalset['겹침'].describe()[['min', '50%', 'max']].round(3).to_string())
print(f"0.5 를 넘는 문항: {(evalset['겹침'] > 0.5).sum()}개")

In [ ]:
# 4번·6번 점검 -- 정답 개수. 하나뿐인 문항만 있어도 안 되고, 지나치게 많아도 안 됩니다.
evalset['정답수'] = evalset['gold_chunks'].str.split('|').apply(len)
print(evalset['정답수'].value_counts().sort_index().to_string())
print(f"복수 정답 문항: {(evalset['정답수'] > 1).sum()}개 / {len(evalset)}문항")

# 6번은 기준을 정해 두어야 판정할 수 있습니다. 여기서는 '정답이 8개를 넘으면 질문이 너무 넓다' 로 봅니다.
too_wide = evalset[evalset['정답수'] > 8]['query_id'].tolist()
print('정답 조각이 8개를 넘는 문항:', too_wide)

In [ ]:
# 5번 점검 -- 지금 검색기가 상위 3개에서 못 찾는 문항이 있는가.
def found_at_k(row, k):
    """상위 k개와 정답의 교집합. 비어 있으면 그 안에 정답이 하나도 없다는 뜻이다."""
    return set(search_ids(store_400, row.query, k)) & set(row.gold_chunks.split('|'))


missed = [row.query_id for row in evalset.itertuples() if not found_at_k(row, 3)]
print(f'상위 3개에서 못 찾은 문항 {len(missed)}개: {missed}')

1번부터 5번까지는 통과했습니다. 6번도 기준(8개)을 넘는 문항이 없어 통과입니다. 다만 정답 개수 분포를 보면 **정답이 5개 이상인 문항이 스무 개 중 여덟 개**입니다. 그 문항들을 열어 보면 하나같이 "점검할 **항목들**", "목적 외로 쓸 수 있는 **경우들**" 처럼 **열거형 질문**입니다. 답이 안내서 여러 쪽에 흩어져 있으니 라벨이 여러 개가 될 수밖에 없습니다. 이런 문항은 K=3 으로 재면 Recall 이 낮게 나올 수밖에 없습니다. 정답이 5개인데 3개까지만 꺼내 보니 다 맞혀도 3/5 가 최대입니다 — 앞 시간에 "Recall 은 정답 개수와 함께 읽는다" 고 한 이유가 바로 이것입니다. 7번은 지표를 실제로 재 봐야 알 수 있는데, 그것은 앞 시간에 이미 했습니다.

> **평가셋을 만드는 쪽에서 할 수 있는 일이 있습니다.** 라벨이 자꾸 불어나는 질문은 대개 **여러 질문을 하나로 묶은 것**입니다. "점검 항목을 알려 줘" 를 "배포 전 보안 점검 항목" 과 "학습 데이터 점검 항목" 으로 쪼개면 각 문항의 정답이 두세 개로 줄고, 그러면 Recall 이 **검색기의 성능을 재는 숫자**가 됩니다. 쪼갤 수 없는 질문이라면 그 문항만 따로 묶어 **더 큰 K 로** 재는 편이 정직합니다.

특히 5번에서 **못 찾는 문항이 남아 있다는 것**이 중요합니다. 전부 맞히는 평가셋은 만점밖에 나오지 않아서 무엇을 바꾸든 점수가 그대로입니다 — 그런 평가셋은 아무것도 알려 주지 못합니다. 검색을 고쳐 볼 때, **그때 움직일 여지가 있어야** 고친 것이 좋은지 나쁜지 판단할 수 있습니다.

> 점검을 통과했다고 좋은 평가셋인 것은 아닙니다. 이 항목들은 **명백한 결함이 없다**는 뜻일 뿐, 질문이 실제 사용자의 질문과 닮았는지는 여전히 사람이 판단할 몫입니다.

### 🖐️ 함께 따라하기 — 점검 항목을 하나 더 만들기

점검 항목을 하나 더 만들어 봅니다. **같은 질문이 두 번 들어가 있지 않은지**, 그리고 **정답 조각이 여러 문항에 중복해 쓰이고 있지는 않은지** 확인하는 항목입니다.

1. `evalset['query']` 에 중복이 있는지 세어 출력하세요.
2. 모든 문항의 정답 조각 id 를 모은 `gold_all` 에서 **두 번 이상** 쓰인 조각 id 를 찾아 출력하세요.

확인할 것: 중복 질문은 0개여야 합니다. 정답 조각이 겹치는 것은 결함이 아닙니다 — 다만 **한 조각이 여러 문항의 답이라면 그 조각만 잘 찾아도 점수가 오르므로** 알고는 있어야 합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) query 열의 중복 개수 세기 (duplicated().sum())
# 2) gold_all 에서 두 번 이상 등장하는 조각 id 찾기 (count 로 세면 된다)

---
## 부록 정리

| 주제 | 한 일 | 남길 것 |
|---|---|---|
| 평가셋 3요소 | 질문 · 정답 라벨 · 근거 문장 | 근거 문장이 있어야 라벨을 나중에 검수할 수 있다 |
| 라벨링 4단계 | 후보 좁히기 → 판정 → 근거 인용 → 부분문자열 검증 | 판정은 **스키마로** 받고, 확정은 **사람이** 한다 |
| 인용 강제 | 모델이 지어낸 근거를 걸러 낸다 | 인용이 조각 안에 없으면 그 라벨은 버린다 |
| 점검 7항목 | 내보내기 전에 기계로 훑는다 | 특히 **전부 맞히는 평가셋은 쓸모가 없다** |

한 가지만 남긴다면: **재는 도구부터 검수하라**는 것입니다. 평가셋이 틀리면 그것으로 잰 점수가 전부 함께 틀립니다. 그런데 화면에는 멀쩡해 보이는 숫자가 찍히기 때문에 알아채기가 어렵습니다.